# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jerovernay/FlyRank-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Scoring population: May, not March.** ML-08/ML-09 fit and validated the archetypes on March;
ML-09 already established the reuse pattern — freeze one K-Means fit on all of March as the
"production" model (`ref_scaler` / `ref_km`, no OOF/GroupKFold), then `.predict()` on a later
month never touched during fitting. May is the latest non-sealed month (June `_sample` stays a
sealed test month), so this queue scores May pages through that frozen March model — the same
mechanics ML-09 used to measure retention, now used for real.

**Tiering is not "sort by O/E lift."** ML-09 found the four archetypes split on two independent
axes — whether the O/E effect clears chance, and whether the label survives a month — and they
don't agree with each other for `Overlooked`. A queue that just sorted by O/E magnitude would
rank `Overlooked` as a middling action item, which overstates confidence in a label that's gone
by the time anyone could act on it. Tiers instead combine evidence strength, label durability,
and (for the two archetypes with no lift evidence either way) how much value is actually at
stake:

| Tier | Archetype | Action | Reason code | Why |
|---|---|---|---|---|
| 1 — act now | Buried | prune/rewrite | `buried_worse_than_chance_durable` | O/E 0.645–0.667, CI excludes 1.0 (two independent runs), **and** 82.4% March→May retention — the most durable label of the four. Two signals agreeing. |
| 2 — re-verify before acting | Overlooked | improve/expand | `overlooked_no_lift_unstable_reverify` | O/E 0.896, CI crosses 1.0 (indistinguishable from chance), **and** only 35.3% retention — most pages have already relabeled to Steady/Long tail by the time this would be actioned. Queue position reflects the label's instability, not a recommendation against ever improving these pages. |
| 3 — protect | Steady performers | protect/monitor | `steady_high_value_no_lift` | O/E 1.059, chance — no evidence further action helps — but by far the highest median impressions/CTR of the four, so there is real value to lose if these slip. |
| 4 — light monitor | Long tail | monitor | `long_tail_low_value_no_lift` | O/E 1.041, chance, near-zero median CTR — lowest value at stake, cheapest tier to leave alone. |

**Within-tier ranking.** Tiers 1, 3, 4 sort by `total_impressions` descending — biggest
opportunity/value first. Tier 2 sorts by `centroid_dist` ascending (distance to its own cluster
centroid) — the pages that most strongly show the "few impressions, unusually high CTR" pattern
`Overlooked` was named for, same logic ML-08 used for its own Overlooked top-K cut.

**A population-eligibility note carried over from ML-09:** `is_published` / `is_deleted` live in
`dim_content`, which is not month-partitioned, so those flags are joined from the March cache
onto May by `content_hash_id`. Pages present in May but absent from March's snapshot (new content)
have no such flag and are excluded from this queue — a real gap, named in Section 2's limits.</invoke>


In [1]:
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message=".*valid feature names.*")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

SEED, VOL_FLOOR, K_CLUSTERS = 42, 100, 4
FEATURES = ["log_impressions", "log_clicks", "ctr", "avg_position"]
OUTD = "../outputs"
os.makedirs(OUTD, exist_ok=True)

MARCH_CACHE = f"{OUTD}/w05_march_features.parquet"
MAY_CACHE = f"{OUTD}/w06_may_features.parquet"
assert os.path.exists(MARCH_CACHE) and os.path.exists(MAY_CACHE), \
    "run w05_model.ipynb and w06_validation_audit.ipynb once first to populate these caches"

# --- Rebuild the March population + the frozen "production" reference model (identical to ML-09) ---
march = pd.read_parquet(MARCH_CACHE)
pop_march = march[march.is_published & (~march.is_deleted)
                   & (march.total_impressions > 0) & (march.avg_position > 0)
                   & (march.total_impressions >= VOL_FLOOR)].copy().reset_index(drop=True)
pop_march["ctr"] = pop_march.total_clicks / pop_march.total_impressions * 100
X_march = pd.DataFrame({"log_impressions": np.log1p(pop_march.total_impressions),
                        "log_clicks": np.log1p(pop_march.total_clicks),
                        "ctr": pop_march.ctr,
                        "avg_position": pop_march.avg_position})[FEATURES]

ref_scaler = StandardScaler().fit(X_march)
ref_km = KMeans(n_clusters=K_CLUSTERS, random_state=SEED, n_init=10).fit(ref_scaler.transform(X_march))
ARCHETYPES = {0: "Steady performers", 1: "Long tail", 2: "Buried", 3: "Overlooked"}
print("reference centroids (original units), matched to ML-08/ML-09's naming:")
print(pd.DataFrame(ref_scaler.inverse_transform(ref_km.cluster_centers_), columns=FEATURES)
      .assign(archetype=lambda d: d.index.map(ARCHETYPES)).round(3).to_string(index=False))

# --- Score May: the latest non-sealed month, through the frozen March model ---
may = pd.read_parquet(MAY_CACHE)
# client_hash_id/is_published/is_deleted live in March's dim_content join (not month-partitioned,
# content_hash_id -> client_hash_id is a verified 1:1 mapping - ML-08's grain probe)
flags = march[["content_hash_id", "client_hash_id", "is_published", "is_deleted"]] \
    .drop_duplicates("content_hash_id")
may = may.merge(flags, on="content_hash_id", how="inner")  # drops May-only content, no flag to check

pop_may = may[may.is_published & (~may.is_deleted)
              & (may.total_impressions > 0) & (may.avg_position > 0)
              & (may.total_impressions >= VOL_FLOOR)].copy().reset_index(drop=True)
pop_may["ctr"] = pop_may.total_clicks / pop_may.total_impressions * 100
X_may = pd.DataFrame({"log_impressions": np.log1p(pop_may.total_impressions),
                      "log_clicks": np.log1p(pop_may.total_clicks),
                      "ctr": pop_may.ctr,
                      "avg_position": pop_may.avg_position})[FEATURES]

Xs_may = ref_scaler.transform(X_may)
pop_may["cluster"] = ref_km.predict(Xs_may)
pop_may["archetype"] = pop_may.cluster.map(ARCHETYPES)
pop_may["centroid_dist"] = np.linalg.norm(Xs_may - ref_km.cluster_centers_[pop_may.cluster], axis=1)

n_may_raw = len(may)
print(f"\nMay rows (post dim_content join): {n_may_raw} -> queue-eligible: {len(pop_may)} "
      f"({len(pop_may)/n_may_raw:.1%})")
print(pop_may.archetype.value_counts().to_string())

# --- Tier / action / reason code ---
TIER_MAP = {
    "Buried":             {"tier": 1, "action": "prune/rewrite",
                            "reason_code": "buried_worse_than_chance_durable"},
    "Overlooked":         {"tier": 2, "action": "improve/expand",
                            "reason_code": "overlooked_no_lift_unstable_reverify"},
    "Steady performers":  {"tier": 3, "action": "protect/monitor",
                            "reason_code": "steady_high_value_no_lift"},
    "Long tail":          {"tier": 4, "action": "monitor",
                            "reason_code": "long_tail_low_value_no_lift"},
}
tier_df = pd.DataFrame(TIER_MAP).T.rename_axis("archetype").reset_index()
queue = pop_may.merge(tier_df, on="archetype", how="left")

# within-tier sort key: centroid_dist (ascending) for tier 2, total_impressions (descending) otherwise
queue["sort_key"] = np.where(queue.tier == 2, queue.centroid_dist, -queue.total_impressions)
queue = queue.sort_values(["tier", "sort_key"]).drop(columns="sort_key")
queue["rank_within_tier"] = queue.groupby("tier").cumcount() + 1
queue["overall_rank"] = np.arange(1, len(queue) + 1)

cols = ["overall_rank", "tier", "rank_within_tier", "archetype", "action", "reason_code",
        "content_hash_id", "client_hash_id", "total_impressions", "total_clicks", "ctr",
        "avg_position", "centroid_dist"]
queue = queue[cols]

print(f"\nranked queue: {len(queue)} pages")
print("\ntier sizes:")
print(queue.groupby(["tier", "archetype", "action"]).size().rename("n").to_string())
print("\ntop 3 rows per tier:")
for t in sorted(queue.tier.unique()):
    print(f"\n--- tier {t} ---")
    print(queue[queue.tier == t].head(3).round(3).to_string(index=False))


reference centroids (original units), matched to ML-08/ML-09's naming:
 log_impressions  log_clicks   ctr  avg_position         archetype
           8.451       2.553 0.328         9.442 Steady performers
           6.159       0.427 0.131        10.309         Long tail
           5.877       0.188 0.060        45.777            Buried
           6.266       2.111 1.400         8.502        Overlooked



May rows (post dim_content join): 331437 -> queue-eligible: 93728 (28.3%)
archetype
Long tail            40813
Steady performers    25110
Buried               20764
Overlooked            7041



ranked queue: 93728 pages

tier sizes:
tier  archetype          action         
1     Buried             prune/rewrite      20764
2     Overlooked         improve/expand      7041
3     Steady performers  protect/monitor    25110
4     Long tail          monitor            40813

top 3 rows per tier:

--- tier 1 ---
 overall_rank tier  rank_within_tier archetype        action                      reason_code          content_hash_id          client_hash_id  total_impressions  total_clicks   ctr  avg_position  centroid_dist
            1    1                 1    Buried prune/rewrite buried_worse_than_chance_durable content_c60628276389acbb client_23a62021009f63c4           160983.0           7.0 0.004        86.293          5.350
            2    1                 2    Buried prune/rewrite buried_worse_than_chance_durable content_a9322b74ca7cb1bb client_23a62021009f63c4           109052.0           1.0 0.001        84.424          4.841
            3    1                 3    Buried p

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** The queue is a monthly prioritization aid for a content ops / SEO reviewer —
"where do we look first," not "what will happen if we act" and not an automation. Tier order
reflects decision-support confidence, not a promise of outcome, and nothing in it should be wired
to auto-execute (Section 3 makes this explicit).

**Limits, grouped by cause:**

1. **No causal claim, anywhere, including the untested part.** Every O/E number behind this queue
   comes from one observed, non-intervention window (March→April). Scoring May and expecting that
   window's dynamics to still apply forward (May→June) is an **extrapolation this notebook has not
   itself re-validated** — the honest position is that April's evidence is the best available
   basis for a May queue, not that it has been re-confirmed on May outcomes.
2. **Evidence strength is uneven by tier, on purpose.** Tier 1 (Buried) rests on two independent
   signals (O/E CI excludes 1.0, 82.4% durability). Tier 2 (Overlooked)'s action has essentially no
   supporting lift evidence (O/E CI crosses 1.0) and the least durable label of the four — its
   queue position is a caution flag, not a strength ranking. Tiers 3–4 have no lift evidence in
   either direction; "protect" vs. "monitor" there is a value-at-stake judgment, not a measured
   effect.
3. **Archetype is a group-level lens, not a per-page guarantee.** ML-09 documented concrete misses
   inside a correct group verdict: a `Buried` page that recovered from zero clicks to 4.4% CTR
   anyway, and a `Steady performers` page that lost most of its traffic and position in the same
   month. A single row's tier says nothing certain about that row.
4. **Population scope is narrow and concentrated.** The archetypes were fit on 44 clients, one of
   which is 21% of the whole modeling population (checked below) — centroids may not transfer well
   to a client very unlike that mix. GA4 on-site engagement is excluded from the feature set
   entirely (ML-08), so nothing here describes on-page behavior. No "ranking lags demand" archetype
   exists in this feature set — plausibly a commercially important group this method cannot see.
5. **Coverage gap.** Most of May's raw inventory never reaches this queue (confirmed below) —
   below the volume floor, missing position data, unpublished/deleted, or absent from March's
   `dim_content` snapshot entirely. This queue has nothing to say about excluded pages, including
   any content newer than March.
6. **This is a snapshot, not a live score.** Section 4 covers what would signal it's gone stale.

In [2]:
# --- Confirm the two numeric claims above: the coverage gap (point 5) and client concentration (point 4) ---
# Sequential funnel: each step's drop count is taken from what SURVIVED the previous step,
# so the four counts are mutually exclusive and sum exactly to the total drop.
raw_may = pd.read_parquet(MAY_CACHE)
n_raw_may = len(raw_may)

step0 = raw_may
step1 = step0.merge(flags, on="content_hash_id", how="inner")           # has March dim_content match
step2 = step1[step1.is_published & (~step1.is_deleted)]                 # published, not deleted
step3 = step2[(step2.total_impressions > 0) & (step2.avg_position > 0)]  # has impressions + position
step4 = step3[step3.total_impressions >= VOL_FLOOR]                     # clears the volume floor

d_no_march_match = len(step0) - len(step1)
d_unpub_or_deleted = len(step1) - len(step2)
d_no_signal = len(step2) - len(step3)
d_below_floor = len(step3) - len(step4)

print("=== coverage gap funnel (point 5) ===")
print(f"May raw rows (fact table, all content):              {len(step0):>7}")
print(f"  dropped - no match in March's dim_content snapshot: {d_no_march_match:>7}")
print(f"  dropped - unpublished or deleted:                   {d_unpub_or_deleted:>7}")
print(f"  dropped - no impressions or no position data:       {d_no_signal:>7}")
print(f"  dropped - below the {VOL_FLOOR}-impression volume floor:      {d_below_floor:>7}")
print(f"queue-eligible (matches Section 1's ranked queue):    {len(step4):>7}")
assert len(step4) == len(pop_may), "funnel end must match Section 1's queue-eligible population"
print(f"\n{len(step4)/len(step0):.1%} of raw May inventory reaches this queue; "
      f"{1 - len(step4)/len(step0):.1%} is out of scope for it.")

print("\n=== client concentration (point 4) ===")
top_client_share = pop_march.client_hash_id.value_counts(normalize=True).iloc[0]
print(f"March modeling population: {pop_march.client_hash_id.nunique()} clients, "
      f"largest single client is {top_client_share:.1%} of all pages "
      f"(matches ML-08's reported 21%).")


=== coverage gap funnel (point 5) ===
May raw rows (fact table, all content):               389153
  dropped - no match in March's dim_content snapshot:   57716
  dropped - unpublished or deleted:                     10331
  dropped - no impressions or no position data:        150729
  dropped - below the 100-impression volume floor:        76649
queue-eligible (matches Section 1's ranked queue):      93728

24.1% of raw May inventory reaches this queue; 75.9% is out of scope for it.

=== client concentration (point 4) ===
March modeling population: 44 clients, largest single client is 21.3% of all pages (matches ML-08's reported 21%).


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Per-tier review checklist.**

- **Tier 1 (Buried, prune/rewrite):** confirm the page isn't newly published or still ramping up
  (no page-age feature exists in this pipeline, so this is a manual check against the CMS), isn't
  legally required or carrying backlink value beyond organic search, and decide prune *vs.*
  rewrite — the model only says "this isn't working," it can't tell those two actions apart.
- **Tier 2 (Overlooked, improve/expand):** re-score the page's *current* archetype before acting.
  With 35.3% March→May retention, a page's May label can already be stale by the time anyone
  reaches it — acting on the frozen snapshot without re-checking is exactly what that instability
  warns against.
- **Tier 3 (Steady performers, protect/monitor):** "protect" needs a human glance at recent
  trend. O/E-at-chance is a group-level statement ("no measured lift from further action"), not a
  promise that any individual page in the tier is currently stable.
- **Tier 4 (Long tail, monitor):** lowest review priority; periodic spot-check only.

**No-go list — never automate:**
- Never auto-delete, auto-unpublish, or auto-rewrite content from the tier/archetype alone — a
  named human reviewer signs off on every executed action, citing the row's `reason_code`.
- Never act on Tier 2 without re-verifying current archetype membership first.
- Never treat one row's tier as certain about that specific page — Section 2 already showed a
  `Buried` page recovering and a `Steady performers` page collapsing inside a correct group
  verdict.
- Never use this queue for cross-client resourcing or budget decisions without accounting for the
  21%-single-client concentration in the population it was fit on.

**The queue is too big to review by tier — checked below, for both large tiers.** Tier 1 alone is
20,764 pages and Tier 3 is 25,110 — neither is a one-cycle review list. The practical fix here is
a per-client cap, checked quantitatively below rather than assumed.

**Future work, not built here.** This playbook treats each archetype as one flat scope with one
action. A natural next layer — out of scope for this notebook, and named here on purpose — is a
second, narrower scoring pass *within* each archetype (e.g., a dedicated sub-lane per archetype,
or an archetype-specific feature set) to rank which pages inside a 20,000-page tier most need
attention first, rather than relying on `total_impressions` as the only tiebreaker. Composing a
coarse group-level lens (this notebook) with a finer per-archetype lens (future work) is a more
honest way to shrink 20k pages to a workable list than picking an arbitrary cap alone.

In [3]:
# --- Review-load check: can Tier 1 / Tier 3 realistically be reviewed as a whole? ---
for t in (1, 3):
    tname = queue.loc[queue.tier == t, "archetype"].iloc[0]
    per_client = queue[queue.tier == t].groupby("client_hash_id").size()
    print(f"=== Tier {t} ({tname}): {per_client.sum()} pages across {per_client.size} clients ===")
    print(f"per-client pages: median {per_client.median():.0f} | mean {per_client.mean():.1f} "
          f"| max {per_client.max()} | p90 {per_client.quantile(0.9):.0f}")
    for cap in (10, 20, 50):
        covered = per_client.clip(upper=cap).sum()
        print(f"  cap at top-{cap}/client/cycle -> review {covered} pages "
              f"({covered/per_client.sum():.1%} of the tier), "
              f"fully covers {(per_client <= cap).mean():.1%} of clients in one pass")
    print()

print("A top-20-per-client-per-cycle cap keeps every cycle's review list in the hundreds, not "
      "tens of thousands, while still touching every client with pages in the tier.")


=== Tier 1 (Buried): 20764 pages across 32 clients ===
per-client pages: median 92 | mean 648.9 | max 4002 | p90 2352
  cap at top-10/client/cycle -> review 268 pages (1.3% of the tier), fully covers 21.9% of clients in one pass
  cap at top-20/client/cycle -> review 505 pages (2.4% of the tier), fully covers 31.2% of clients in one pass
  cap at top-50/client/cycle -> review 1099 pages (5.3% of the tier), fully covers 43.8% of clients in one pass

=== Tier 3 (Steady performers): 25110 pages across 34 clients ===
per-client pages: median 54 | mean 738.5 | max 7337 | p90 2232
  cap at top-10/client/cycle -> review 284 pages (1.1% of the tier), fully covers 32.4% of clients in one pass
  cap at top-20/client/cycle -> review 498 pages (2.0% of the tier), fully covers 41.2% of clients in one pass
  cap at top-50/client/cycle -> review 1067 pages (4.2% of the tier), fully covers 47.1% of clients in one pass

A top-20-per-client-per-cycle cap keeps every cycle's review list in the hundreds, 

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**No model is persisted anywhere in this project** — K-Means isn't pickled in ML-08 or ML-09,
only printed centroid numbers survive as the record. So "retrain" here means re-running the
notebook chain (`w05_model` → `w06_validation_audit` → this queue) on a fresh base month, not an
automated pipeline swap. That is a deliberate, non-production choice, not an oversight.

**Four triggers, each tied to a number already established in this project:**

1. **Archetype retention drift.** ML-09 measured March→May retention (65.8% overall, chance
   baseline 33.0%, per-archetype 82.4% down to 35.3%). Recomputed below, reusing `pop_march` /
   `pop_may` / `ref_km` already built in Section 1 — this notebook's own baseline to compare
   future cycles against. **Trigger: if a future cycle's overall retention falls below the
   midpoint between the baseline and the chance rate, or `Buried`'s retention specifically drops
   toward its own chance share, the archetype structure is destabilizing — re-run everything.**
   `Buried` gets its own named check because it carries Tier 1's entire justification.
2. **Coverage/funnel drift.** Section 2's funnel (24.1% of raw May reaches the queue) should stay
   roughly stable cycle to cycle. A sharp shift signals a pipeline change or a real inventory
   shift — investigate before trusting the queue, don't just re-run it.
3. **Client concentration drift.** If the dominant client's share (21.3%, Section 2/3) grows
   further, or a new client comes to dominate, the "one archetype set fits every client"
   assumption gets weaker — ties to Section 3's future-work note on per-archetype sub-scoring.
4. **O/E re-validation.** Once a future outcome window opens, the O/E audit (ML-08/09's core
   evidence) should be re-run against it. If `Buried`'s O/E confidence interval ever crosses 1.0,
   Tier 1's whole justification collapses — the highest-stakes trigger of the four, and the one
   that most directly overturns the playbook rather than just aging it.

In [4]:
# --- Recompute March->May archetype retention, this notebook's own baseline for trigger 1 ---
pop_march_arch = pop_march.copy()
pop_march_arch["archetype"] = ref_km.labels_  # ref_km was fit directly on X_march, same row order
pop_march_arch["archetype"] = pop_march_arch.archetype.map(ARCHETYPES)

inter = pop_march_arch[["content_hash_id", "archetype"]].merge(
    pop_may[["content_hash_id", "archetype"]], on="content_hash_id",
    suffixes=("_march", "_may"))

overall_retention = (inter.archetype_march == inter.archetype_may).mean()
march_share = inter.archetype_march.value_counts(normalize=True)
may_share = inter.archetype_may.value_counts(normalize=True).reindex(march_share.index, fill_value=0)
chance = (march_share * may_share).sum()  # overall chance: weighted avg of per-archetype chance below

conf = pd.crosstab(inter.archetype_march, inter.archetype_may, normalize="index") \
         .reindex(index=march_share.index, columns=march_share.index, fill_value=0)
per_arch = pd.Series(np.diag(conf), index=conf.index)
# per-archetype chance = P(May=a) alone, independent of where a page started in March -
# NOT march_share*may_share (that's the joint probability, not the conditional retention rate)
per_arch_chance = may_share

print(f"=== March -> May retention, recomputed here (n={len(inter)}) ===")
print(f"overall retention: {overall_retention:.1%}  (ML-09 reported 65.8%)")
print(f"chance baseline:    {chance:.1%}  (ML-09 reported 33.0%)")
print("\nper-archetype retention (this run vs ML-09's baseline):")
ml09_baseline = {"Buried": 0.824, "Long tail": 0.671, "Steady performers": 0.648, "Overlooked": 0.353}
for a in per_arch.index:
    print(f"  {a:<20} {per_arch[a]:.1%}  (ML-09: {ml09_baseline[a]:.1%})  "
          f"chance {per_arch_chance[a]:.1%}")

# --- Concrete trigger thresholds: midpoint between each baseline and its own chance rate ---
print("\n=== retrain-trigger thresholds (midpoint of baseline and chance) ===")
overall_trigger = (overall_retention + chance) / 2
print(f"overall retention below {overall_trigger:.1%} -> retrain")
for a in per_arch.index:
    t = (per_arch[a] + per_arch_chance[a]) / 2
    flag = "  <- Buried carries Tier 1's justification" if a == "Buried" else ""
    print(f"  {a:<20} retention below {t:.1%} -> retrain{flag}")


=== March -> May retention, recomputed here (n=81547) ===
overall retention: 65.8%  (ML-09 reported 65.8%)
chance baseline:    33.0%  (ML-09 reported 33.0%)

per-archetype retention (this run vs ML-09's baseline):
  Long tail            67.1%  (ML-09: 67.1%)  chance 43.0%
  Steady performers    64.8%  (ML-09: 64.8%)  chance 29.1%
  Buried               82.4%  (ML-09: 82.4%)  chance 20.7%
  Overlooked           35.3%  (ML-09: 35.3%)  chance 7.2%

=== retrain-trigger thresholds (midpoint of baseline and chance) ===
overall retention below 49.4% -> retrain
  Long tail            retention below 55.0% -> retrain
  Steady performers    retention below 46.9% -> retrain
  Buried               retention below 51.6% -> retrain  <- Buried carries Tier 1's justification
  Overlooked           retention below 21.3% -> retrain


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Three exports, checked against `.gitignore` before writing anything:

1. **`work/outputs/ranked_action_queue.csv`** — the full Section 1 queue. `.gitignore` blocks
   every CSV under `work/` on purpose (the leak guard), so this stays out of git and is
   regenerated by re-running this notebook — matching the assignment's own note that the queue
   CSV is not meant to be a committed artifact.
2. **`work/outputs/ml10_playbook_metrics.json`** (committed — no matching ignore rule) — the
   single receipts file every number in Sections 1–4 traces back to: tier sizes/actions/reason
   codes, the O/E and retention evidence behind each tier, the coverage funnel, client
   concentration, per-client review caps, and the Section 4 retrain-trigger thresholds.
3. **Two figures to `work/figures/`** (committed): a retention-vs-chance chart per archetype (the
   visual case for the tiering logic) and a tier-size chart (why a per-client review cap is
   necessary). Archetype identity (which bar is Buried vs. Overlooked vs. Steady vs. Long tail)
   uses one fixed CVD-validated categorical color per archetype — blue/orange/aqua/yellow, the
   same mapping used in the capstone paper's archetype-profile figure — in **both** figures now,
   not just the tier-size one: color should follow the entity (the archetype), not its current
   rank or which chart it's in, so the same archetype reads as the same color everywhere it
   appears in the paper. The retention chart's "chance baseline" bars stay a single neutral gray
   (`#898781`) — that series is a reference line, not a 5th archetype, so it doesn't get a
   categorical slot.

In [5]:
import json
import matplotlib.pyplot as plt

FIGD = "../figures"
os.makedirs(FIGD, exist_ok=True)

# --- 1. Ranked queue CSV (gitignored, regenerated) ---
queue_path = f"{OUTD}/ranked_action_queue.csv"
queue.to_csv(queue_path, index=False)
print(f"wrote {queue_path}  ({len(queue)} rows)")

# --- 2. Metrics JSON: the receipts every number above traces back to (committed) ---
metrics = {
    "scoring_month": "2026-05", "reference_fit_month": "2026-03",
    "tiers": [
        {"tier": 1, "archetype": "Buried", "action": "prune/rewrite",
         "reason_code": "buried_worse_than_chance_durable", "n_pages": int((queue.tier == 1).sum()),
         "oe_lift": {"ml08": [0.645, 0.456, 0.810], "ml09_recheck": [0.667, 0.470, 0.855]},
         "march_may_retention": 0.824, "review_cap_top20_per_client":
             int(queue[queue.tier == 1].groupby("client_hash_id").size().clip(upper=20).sum())},
        {"tier": 2, "archetype": "Overlooked", "action": "improve/expand",
         "reason_code": "overlooked_no_lift_unstable_reverify", "n_pages": int((queue.tier == 2).sum()),
         "oe_lift": {"ml08": [0.896, 0.714, 1.050]}, "march_may_retention": 0.353},
        {"tier": 3, "archetype": "Steady performers", "action": "protect/monitor",
         "reason_code": "steady_high_value_no_lift", "n_pages": int((queue.tier == 3).sum()),
         "oe_lift": {"ml08": [1.059, 0.865, 1.233]}, "march_may_retention": 0.648,
         "review_cap_top20_per_client":
             int(queue[queue.tier == 3].groupby("client_hash_id").size().clip(upper=20).sum())},
        {"tier": 4, "archetype": "Long tail", "action": "monitor",
         "reason_code": "long_tail_low_value_no_lift", "n_pages": int((queue.tier == 4).sum()),
         "oe_lift": {"ml08": [1.041, 0.930, 1.150]}, "march_may_retention": 0.671},
    ],
    "coverage_funnel_may": {
        "raw_rows": int(len(step0)), "dropped_no_march_match": int(d_no_march_match),
        "dropped_unpublished_or_deleted": int(d_unpub_or_deleted),
        "dropped_no_signal": int(d_no_signal), "dropped_below_volume_floor": int(d_below_floor),
        "queue_eligible": int(len(step4)), "coverage_pct": round(len(step4) / len(step0), 4),
    },
    "client_concentration": {"n_clients": int(pop_march.client_hash_id.nunique()),
                              "top_client_share": round(float(top_client_share), 4)},
    "retrain_triggers": {
        "overall_retention_baseline": round(float(overall_retention), 4),
        "overall_chance": round(float(chance), 4),
        "overall_trigger_below": round(float(overall_trigger), 4),
        "per_archetype": {a: {"baseline": round(float(per_arch[a]), 4),
                              "chance": round(float(per_arch_chance[a]), 4),
                              "trigger_below": round(float((per_arch[a] + per_arch_chance[a]) / 2), 4)}
                          for a in per_arch.index},
    },
}
metrics_path = f"{OUTD}/ml10_playbook_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"wrote {metrics_path}")

# --- Shared archetype palette (dataviz skill palette.md slots 1-4, CVD-validated) ---
# One fixed color per archetype identity, reused across every figure in this notebook and the
# capstone paper's archetype-profile figure -- color follows the entity (the archetype), not its
# current tier rank. Validated: node scripts/validate_palette.js
# "#2a78d6,#eb6834,#1baf7a,#eda100" --mode light (and dark steps).
archetype_ramp = {"Buried": "#2a78d6", "Overlooked": "#eb6834",
                   "Steady performers": "#1baf7a", "Long tail": "#eda100"}

# --- 3a. Figure: retention vs chance, per archetype ---
BLUE, GRAY, INK, SEC_INK, MUTED, GRID, BASE, SURF = \
    "#2a78d6", "#898781", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
order = ["Buried", "Overlooked", "Steady performers", "Long tail"]  # tier priority order (1->4)
retention_vals = [per_arch[a] for a in order]
chance_vals = [per_arch_chance[a] for a in order]

fig, ax = plt.subplots(figsize=(7, 4.5), facecolor=SURF)
ax.set_facecolor(SURF)
x = np.arange(len(order))
w = 0.34
# Observed bars: one per-archetype color each (same mapping as archetype_profile.png /
# tier_sizes.png), so the same archetype reads as the same color throughout the paper. The x-axis
# already names each archetype directly, so this is reinforcement, not the only identity cue.
# Chance bars stay a single neutral gray -- it's a reference baseline, not a 5th entity, so it
# doesn't get a slot in the categorical palette.
b1 = ax.bar(x - w / 2, retention_vals, w, color=[archetype_ramp[a] for a in order])
b2 = ax.bar(x + w / 2, chance_vals, w, color=GRAY, label="chance baseline (independent relabel)")
for bars in (b1, b2):
    for r in bars:
        ax.text(r.get_x() + r.get_width() / 2, r.get_height() + 0.015, f"{r.get_height():.0%}",
                ha="center", va="bottom", fontsize=9, color=INK)
ax.set_xticks(x); ax.set_xticklabels(order, color=SEC_INK)
ax.set_ylim(0, 1.0)
ax.set_ylabel("retention rate", color=SEC_INK)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(BASE)
ax.tick_params(colors=MUTED, length=0)
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.14), fontsize=9)
ax.set_title("Archetype label durability, March → May", color=INK, fontsize=12, pad=44)
fig.text(0.01, -0.06, "Colored bar (left) = observed retention, colored per archetype. Gray bar (right) = chance baseline.",
          fontsize=8.8, color=SEC_INK, ha="left", va="top")
fig.tight_layout()
fig1_path = f"{FIGD}/archetype_retention_vs_chance.png"
fig.savefig(fig1_path, dpi=150, facecolor=SURF, bbox_inches="tight")
plt.close(fig)
print(f"wrote {fig1_path}")

# --- 3b. Figure: tier sizes ---
tier_archetype = {t: queue.loc[queue.tier == t, "archetype"].iloc[0] for t in (1, 2, 3, 4)}
tier_labels = [f"Tier {t}\n{tier_archetype[t]}" for t in (1, 2, 3, 4)]
tier_sizes = [int((queue.tier == t).sum()) for t in (1, 2, 3, 4)]

fig, ax = plt.subplots(figsize=(7, 4.2), facecolor=SURF)
ax.set_facecolor(SURF)
bars = ax.bar(tier_labels, tier_sizes,
              color=[archetype_ramp[tier_archetype[t]] for t in (1, 2, 3, 4)], width=0.6)
for r, n in zip(bars, tier_sizes):
    ax.text(r.get_x() + r.get_width() / 2, r.get_height() + max(tier_sizes) * 0.015, f"{n:,}",
            ha="center", va="bottom", fontsize=9, color=INK)
ax.set_ylabel("pages in queue", color=SEC_INK)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(BASE)
ax.tick_params(colors=MUTED, length=0)
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.set_title("Queue size by tier — too large to review whole, hence the per-client cap",
             color=INK, fontsize=11, pad=14)
fig.tight_layout()
fig2_path = f"{FIGD}/tier_sizes.png"
fig.savefig(fig2_path, dpi=150, facecolor=SURF)
plt.close(fig)
print(f"wrote {fig2_path}")


wrote ../outputs/ranked_action_queue.csv  (93728 rows)


wrote ../outputs/ml10_playbook_metrics.json


wrote ../figures/archetype_retention_vs_chance.png


wrote ../figures/tier_sizes.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Reproducibility.** Seed fixed at 42 (`KMeans`). Warehouse scans are cached to
`work/outputs/*.parquet` (gitignored, reused from `w05_model.ipynb` / `w06_validation_audit.ipynb`
without needing a fresh Hugging Face pull). The reference K-Means model is refit here from
`w05_march_features.parquet` rather than loaded from a pickle — no model object is persisted
anywhere in this project, only printed centroids and the receipts in
`work/outputs/ml10_playbook_metrics.json`.</cell id="dc6f4587">
